In [4]:
import numpy as np, joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

In [5]:
df_full = pd.read_csv(
    "../Dataset.csv",
    parse_dates=['Order_Date']
)
df_sales = df_full[df_full['Is_Return']=='No'].copy()

In [6]:
# Weekly aggregation
df_sales['Week_Start'] = (
    df_sales['Order_Date']
    - pd.to_timedelta(df_sales['Order_Date'].dt.weekday, unit='D')
)
weekly = (df_sales
          .groupby('Week_Start')['Net_Sales_Value']
          .sum().reset_index()
          .sort_values('Week_Start').reset_index(drop=True))

print(f"Weekly points: {len(weekly)}") 

Weekly points: 105


In [7]:
# Scale
values   = weekly['Net_Sales_Value'].values.reshape(-1,1)
scaler_y = MinMaxScaler()
scaled   = scaler_y.fit_transform(values)
joblib.dump(scaler_y, '../models/scaler_y.pkl')

['../models/scaler_y.pkl']

In [8]:
# Sliding window sequences (input=8 weeks → predict week 9)
WINDOW = 8
X_seq, y_seq = [], []
for i in range(WINDOW, len(scaled)):
    X_seq.append(scaled[i-WINDOW:i, 0])
    y_seq.append(scaled[i, 0])

X_seq = np.array(X_seq).reshape(-1, WINDOW, 1)
y_seq = np.array(y_seq)

In [9]:
# Chronological split
split = int(len(X_seq) * 0.80)
X_tr, X_te = X_seq[:split], X_seq[split:]
y_tr, y_te = y_seq[:split], y_seq[split:]
print(f"Train seqs: {len(X_tr)} | Test seqs: {len(X_te)}")

Train seqs: 77 | Test seqs: 20
